# processing large ensembles

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
# import matplotlib.pyplot as plt
# from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
# import matplotlib as mpl
# import matplotlib.ticker as mticker
import netCDF4
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.util import add_cyclic_point
# from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [4]:
# from dask.distributed import Client
# client = Client(n_workers=4, threads_per_worker=2)
# client

In [5]:
from scipy import stats

In [6]:
# mpl.rcParams['font.family'] = 'Droid Sans'
# mpl.rcParams['font.size'] = 12
# # Edit axes parameters
# mpl.rcParams['axes.linewidth'] = 1.5
# # Tick properties
# mpl.rcParams['xtick.major.size'] = 5
# mpl.rcParams['xtick.minor.size'] = 3
# mpl.rcParams['xtick.major.width'] = 1
# mpl.rcParams['xtick.direction'] = 'out'
# mpl.rcParams['ytick.major.size'] = 5
# mpl.rcParams['ytick.minor.size'] = 3
# mpl.rcParams['ytick.major.width'] = 1
# mpl.rcParams['ytick.direction'] = 'out'

In [7]:
from dask.diagnostics import ProgressBar

### Functions needed for the analysis

In [8]:
from functions import preproc_funcs as funcs

In [9]:
from functions import xr_lowess

In [10]:
from statsmodels.tsa.seasonal import STL


def loess1d(x, period):
    x_copy = x.copy()
    res = STL(x_copy, period=period).fit()
    return res.trend


def loess3d(x, dim, period):
    return xr.apply_ufunc(loess1d, x, input_core_dims=[[dim]], output_core_dims=[[dim]], kwargs=dict(period=period), vectorize=True, dask="parallelized")

In [11]:
import glob
import multiprocessing as mp

In [12]:
# Function to find the first file in each model's r1* directory
def find_all_files(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        # model_identifier = path_parts[7] + '_' + path_parts[9][:-1]
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files


# Function to find the first file in each model's r1* directory
def find_all_files_extended(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files



## find the files for a single model

In [ ]:
model = 'MPI-ESM'

In [ ]:
import re

In [14]:
## cmip6
ts_pattern_hist = f'/g/data/**/**/CMIP6/CMIP/**/{model}/historical/**/Amon/ts/gn/latest/*.nc'
ts_pattern_ssp2 = f'/g/data/**/**/CMIP6/ScenarioMIP/**/{model}/ssp245/**/Amon/ts/gn/latest/*.nc'
ts_pattern_ssp5 = f'/g/data/**/**/CMIP6/ScenarioMIP/**/{model}/ssp585/**/Amon/ts/gn/latest/*.nc'

## cmip5
# ts_pattern_rcp85 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_historical_rcp85_*.nc'

In [15]:
# ## cmip6
pr_pattern_hist = f'/g/data/**/**/CMIP6/CMIP/**/{model}/historical/**/Amon/pr/gn/latest/*.nc'
pr_pattern_ssp2 = f'/g/data/**/**/CMIP6/ScenarioMIP/**/{model}/ssp245/**/Amon/pr/gn/latest/*.nc'
pr_pattern_ssp5 = f'/g/data/**/**/CMIP6/ScenarioMIP/**/{model}/ssp585/**/Amon/pr/gn/latest/*.nc'

## cmip5
# pr_pattern_rcp85 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_historical_rcp85_*.nc'

In [16]:
ts_hist_files = sorted(glob.glob(ts_pattern_hist))
ts_ssp2_files = sorted(glob.glob(ts_pattern_ssp2))
ts_ssp5_files = sorted(glob.glob(ts_pattern_ssp5))

# ts_rcp85_files = sorted(glob.glob(ts_pattern_rcp85))

In [17]:
pr_hist_files = sorted(glob.glob(pr_pattern_hist))
pr_ssp2_files = sorted(glob.glob(pr_pattern_ssp2))
pr_ssp5_files = sorted(glob.glob(pr_pattern_ssp5))

# pr_rcp85_files = sorted(glob.glob(pr_pattern_rcp85))

In [18]:
# assert len(pr_hist_files) == len(pr_ssp2_files)

In [19]:
# len(pr_ssp5_files), pr_ssp5_files

In [28]:
def open_member_cmip6(hist_file, ssp_file, ens_idx):
    ds_hist = xr.open_mfdataset(hist_file, use_cftime=True)
    ds_ssp = xr.open_mfdataset(ssp_file, use_cftime=True)

    # concat in time
    ds = xr.concat([ds_hist, ds_ssp], dim='time')

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds


def open_member_cmip6_alternate(ssp_file, ens_idx):
    ds = xr.open_mfdataset(ssp_file, use_cftime=True)

    # concat in time

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds


def open_member_cmip5(rcp_file, ens_idx):
    ds = xr.open_mfdataset(rcp_file, use_cftime=True)

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds

In [29]:
members = []
for i, (hf, sf) in enumerate(zip(ts_hist_files, ts_ssp5_files)):
    members.append(open_member_cmip6(hf, sf, i))


# members = []
# for i, rf in enumerate(ts_ssp3_files):
#     members.append(open_member_cmip6_alternate(rf, i))

ERROR 1: PROJ: proj_create_from_database: Open of /g/data/xp65/public/./apps/med_conda/envs/analysis3-25.09/share/proj failed


In [30]:
ds = xr.concat(members, dim='model')
ds

<xarray.Dataset> Size: 7GB
Dimensions:    (model: 11, time: 5412, bnds: 2, lat: 144, lon: 192)
Coordinates:
  * time       (time) object 43kB 1850-01-16 12:00:00 ... 2300-12-16 12:00:00
  * lat        (lat) float64 1kB -89.38 -88.12 -86.88 ... 86.88 88.12 89.38
  * lon        (lon) float64 2kB 0.9375 2.812 4.688 6.562 ... 355.3 357.2 359.1
  * model      (model) int64 88B 0 1 2 3 4 5 6 7 8 9 10
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (model, time, bnds) object 953kB dask.array<chunksize=(1, 1, 2), meta=np.ndarray>
    lat_bnds   (model, time, lat, bnds) float64 137MB dask.array<chunksize=(1, 906, 144, 2), meta=np.ndarray>
    lon_bnds   (model, time, lon, bnds) float64 183MB dask.array<chunksize=(1, 906, 192, 2), meta=np.ndarray>
    ts         (model, time, lat, lon) float32 7GB dask.array<chunksize=(1, 1, 144, 192), meta=np.ndarray>
Attributes: (12/48)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    branch_method:          standard
    branch_time_in_child:   0.0
    branch_time_in_parent:  164359.0
    contact:                access_csiro@csiro.au
    ...                     ...
    variable_id:            ts
    variant_label:          r10i1p1f1
    version:                v20220819
    license:                CMIP6 model data produced by CSIRO is licensed un...
    cmor_version:           3.4.0
    tracking_id:            hdl:21.14100/995c008b-f08d-42de-bb8d-03967e569dc7

In [32]:
ts = ds["ts"]
# tas = tas.sel(time = slice('1850-01-01', '2100-12-31'))
ts = ts.chunk(dict(model=8, time=120, lat=50, lon=50))

In [33]:
ts_ann = ts.resample(time = 'AS-JUN').mean('time')

In [34]:
with ProgressBar():
    ts_ann_loaded = ts_ann.load()

[########################################] | 100% Completed | 417.22 s


In [ ]:
members = []
for i, (hf, sf) in enumerate(zip(pr_hist_files, pr_ssp5_files)):
    members.append(open_member_cmip6(hf, sf, i))


# members = []
# for i, rf in enumerate(pr_ssp3_files):
#     members.append(open_member_cmip6_alternate(rf, i))

In [24]:
ds = xr.concat(members, dim='model')
ds

<xarray.Dataset> Size: 25GB
Dimensions:    (model: 100, time: 3012, bnds: 2, lat: 72, lon: 144)
Coordinates:
  * time       (time) object 24kB 1850-01-01 00:00:00 ... 2100-12-01 00:00:00
  * lon        (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
  * lat        (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * model      (model) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (model, time, bnds) object 5MB dask.array<chunksize=(1, 1, 2), meta=np.ndarray>
    pr         (model, time, lat, lon) float64 25GB dask.array<chunksize=(1, 1, 72, 144), meta=np.ndarray>
Attributes:
    CDI:               Climate Data Interface version 2.3.0 (https://mpimet.m...
    Conventions:       CF-1.0
    source:            CAM
    case:              b.e21.BHISTcmip6.f09_g17.LE2-1001.001
    logname:           sunseon
    host:              mom1
    initial_file:      b.e21.B1850.f09_g17.CMIP6-piControl.001.cam.i.1001-01-...
    topography_file:   /mnt/lustre/share/CESM/cesm_input/atm/cam/topo/fv_0.9x...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  month_1
    history:           Mon Dec 02 15:54:26 2024: cdo -setattribute,pr@long_na...
    CDO:               Climate Data Operators version 2.3.0 (https://mpimet.m...

In [25]:
pr = ds["pr"]
# pr = pr.sel(time = slice('1850-01-01', '2100-12-31'))
pr = pr.chunk(dict(model=8, time=120, lat=50, lon=50))

In [26]:
pr_ann = pr.resample(time = 'AS-JUN').mean('time')

In [27]:
with ProgressBar():
    pr_ann_loaded = pr_ann.load()

[########################################] | 100% Completed | 18m 30s


In [28]:
pr_ann_loaded

<xarray.DataArray 'pr' (model: 100, time: 252, lat: 72, lon: 144)> Size: 2GB
array([[[[2.04622447e-06, 2.05008962e-06, 2.06912344e-06, ...,
          2.05621735e-06, 2.05504413e-06, 2.04832852e-06],
         [1.68494602e-06, 1.64364679e-06, 1.68604285e-06, ...,
          1.77113267e-06, 1.78529338e-06, 1.75954707e-06],
         [1.80509711e-06, 1.74917474e-06, 1.62169971e-06, ...,
          2.06306090e-06, 2.04368665e-06, 1.91440804e-06],
         ...,
         [1.00166226e-05, 1.00051313e-05, 9.97098715e-06, ...,
          9.41277492e-06, 9.84226514e-06, 1.00525253e-05],
         [8.73650956e-06, 8.83460163e-06, 8.88411726e-06, ...,
          8.06352531e-06, 8.29080014e-06, 8.55117683e-06],
         [6.41491074e-06, 6.43731530e-06, 6.47477583e-06, ...,
          6.23550829e-06, 6.28658743e-06, 6.36617674e-06]],

        [[2.01320866e-06, 2.01023272e-06, 2.00644941e-06, ...,
          2.03101492e-06, 2.02896414e-06, 2.01885287e-06],
         [1.63820248e-06, 1.63361883e-06, 1.62821200e-06, ...,
          1.62999135e-06, 1.62132008e-06, 1.63192690e-06],
         [1.85776356e-06, 1.77834624e-06, 1.64086374e-06, ...,
          2.29125720e-06, 2.16454251e-06, 1.98787366e-06],
...
         [9.46059612e-06, 9.54742041e-06, 9.65706119e-06, ...,
          9.63953584e-06, 9.42795347e-06, 9.32557409e-06],
         [8.17793174e-06, 8.19571575e-06, 8.25609710e-06, ...,
          8.14033827e-06, 8.14847190e-06, 8.15280197e-06],
         [1.00053825e-05, 1.00454403e-05, 1.01034352e-05, ...,
          9.74309718e-06, 9.81913191e-06, 9.92650340e-06]],

        [[3.37669554e-06, 3.37179952e-06, 3.36268044e-06, ...,
          3.40001243e-06, 3.39542461e-06, 3.38624744e-06],
         [2.24852955e-06, 2.22708636e-06, 2.19490480e-06, ...,
          2.36616909e-06, 2.31509990e-06, 2.27662828e-06],
         [1.97644234e-06, 1.91334572e-06, 1.80813219e-06, ...,
          2.32307571e-06, 2.22006715e-06, 2.07799657e-06],
         ...,
         [1.74997136e-05, 1.76102175e-05, 1.79787740e-05, ...,
          1.83023769e-05, 1.79089682e-05, 1.75802270e-05],
         [1.65063480e-05, 1.65679778e-05, 1.66620712e-05, ...,
          1.63644792e-05, 1.64611895e-05, 1.65156445e-05],
         [1.44422221e-05, 1.44169146e-05, 1.43752241e-05, ...,
          1.45124377e-05, 1.45196126e-05, 1.44942504e-05]]]])
Coordinates:
  * lon      (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
  * lat      (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * model    (model) int64 800B 0 1 2 3 4 5 6 7 8 ... 91 92 93 94 95 96 97 98 99
  * time     (time) object 2kB 1849-06-01 00:00:00 ... 2100-06-01 00:00:00
Attributes:
    long_name:     precipitation_flux
    units:         kg m-2 s-1
    cell_methods:  time: mean

In [35]:
ts_ann_loaded.to_netcdf(f'/g/data/if69/as8561/data/enso_tele_robustness/lens/ssp585/{model}_ts.nc')
# pr_ann_loaded.to_netcdf(f'/g/data/if69/as8561/data/enso_tele_robustness/smiles/{model}_pr.nc')

In [139]:
# tas_anom = funcs.calc_anom(tas_ann_loaded, tas_ann_loaded.sel(time = slice('1960', '1990')))
# tas_det = funcs.detrend_rolling_window(tas_anom, window_size=15)

In [140]:
# pr_det = funcs.calc_anom(pr_ann_loaded, pr_ann_loaded.sel(time = slice('1960', '1990')))
# pr_det = funcs.detrend_rolling_window(pr_det, window_size=15)
# pr_std = pr_det/pr_det.std('time')